# Practical Mini-Projects

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/06-spaCy-Linguistic-Practical-Mini-Projects.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/06-spaCy-Linguistic-Practical-Mini-Projects.ipynb)
[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/06-spaCy-Linguistic-Practical-Mini-Projects.ipynb)


## Learning Objectives
By the end of this notebook, you will be able to:
- Build a text summarization helper
- Create an information extraction system
- Develop a text preprocessing pipeline
- Apply spaCy to real-world problems

## Table of Contents
1. [Project 1: Text Summarization Helper](#project-1-text-summarization-helper)
2. [Project 2: Information Extraction](#project-2-information-extraction)
3. [Project 3: Text Preprocessing Pipeline](#project-3-text-preprocessing-pipeline)
4. [Summary and Key Takeaways](#summary-and-key-takeaways)


In [ ]:
# Environment Detection and Setup
import sys
import subprocess
import os

# Detect the runtime environment
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

print(f"Environment detected:")
print(f"  - Local: {IS_LOCAL}")
print(f"  - Google Colab: {IS_COLAB}")
print(f"  - Kaggle: {IS_KAGGLE}")

In [ ]:
# Platform-specific spaCy installation and setup
if IS_COLAB:
    print("\nSetting up Google Colab environment...")
    !pip install -q spacy
    !python -m spacy download en_core_web_sm
elif IS_KAGGLE:
    print("\nSetting up Kaggle environment...")
    # Kaggle usually has spaCy pre-installed
    !python -m spacy download en_core_web_sm
else:
    print("\nSetting up local environment...")
    # For local environment, packages should be installed via requirements.txt
    # Verify spaCy model is available
    try:
        import spacy
        nlp = spacy.load("en_core_web_sm")
        print("✓ spaCy and en_core_web_sm model are available")
    except:
        print("⚠ Please run: python -m spacy download en_core_web_sm")

In [ ]:
import spacy
from spacy import displacy
import re
from collections import Counter

# Load the English language model
nlp = spacy.load("en_core_web_sm")


## Project 1: Text Summarization Helper

### Overview
Build a system that extracts key information from text to help with summarization by identifying:
- Important entities (people, organizations, locations)
- Key noun phrases and concepts
- Important sentences based on entity density


In [ ]:
# Text Summarization Helper
def extract_key_entities(text):
    """Extract important entities from text"""
    doc = nlp(text)
    
    entities = {
        'people': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }
    
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            entities['people'].append(ent.text)
        elif ent.label_ == "ORG":
            entities['organizations'].append(ent.text)
        elif ent.label_ in ["GPE", "LOC"]:
            entities['locations'].append(ent.text)
        elif ent.label_ == "DATE":
            entities['dates'].append(ent.text)
        elif ent.label_ == "MONEY":
            entities['money'].append(ent.text)
    
    return entities

# Test with sample text
sample_text = """
Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in April 1976 in Cupertino, California. 
The company is now worth over $3 trillion and employs more than 150,000 people worldwide. 
Tim Cook has been the CEO since August 2011, succeeding Steve Jobs.
"""

entities = extract_key_entities(sample_text)
print("Key Entities Found:")
print("=" * 25)
for category, items in entities.items():
    if items:
        print(f"{category.title()}: {', '.join(set(items))}")


In [ ]:
def extract_noun_phrases(text):
    """Extract important noun phrases from text"""
    doc = nlp(text)
    
    noun_phrases = []
    for chunk in doc.noun_chunks:
        # Filter out very short or very long phrases
        if 2 <= len(chunk.text.split()) <= 5:
            noun_phrases.append(chunk.text)
    
    return noun_phrases

def score_sentences_by_entities(text):
    """Score sentences based on entity density"""
    doc = nlp(text)
    
    sentences = list(doc.sents)
    scored_sentences = []
    
    for sent in sentences:
        entity_count = len([ent for ent in sent.ents])
        word_count = len([token for token in sent if not token.is_space and not token.is_punct])
        entity_density = entity_count / word_count if word_count > 0 else 0
        
        scored_sentences.append({
            'sentence': sent.text.strip(),
            'entity_count': entity_count,
            'entity_density': entity_density
        })
    
    # Sort by entity density (descending)
    scored_sentences.sort(key=lambda x: x['entity_density'], reverse=True)
    return scored_sentences

# Test the functions
noun_phrases = extract_noun_phrases(sample_text)
print(f"\nKey Noun Phrases:")
print("=" * 20)
for phrase in noun_phrases[:10]:  # Show top 10
    print(f"- {phrase}")

scored_sentences = score_sentences_by_entities(sample_text)
print(f"\nSentences ranked by entity density:")
print("=" * 40)
for i, sent_info in enumerate(scored_sentences, 1):
    print(f"{i}. {sent_info['sentence']}")
    print(f"   Entities: {sent_info['entity_count']}, Density: {sent_info['entity_density']:.3f}")
    print()


## Project 2: Information Extraction System

### Overview
Create a system that extracts structured information from unstructured text, including:
- Person-Organization relationships
- Location-based information
- Temporal information
- Key facts and relationships


In [ ]:
# Information Extraction System
def extract_person_org_relationships(text):
    """Extract relationships between people and organizations"""
    doc = nlp(text)
    
    relationships = []
    sentences = list(doc.sents)
    
    for sent in sentences:
        people = [ent for ent in sent.ents if ent.label_ == "PERSON"]
        orgs = [ent for ent in sent.ents if ent.label_ == "ORG"]
        
        # Find relationships within the same sentence
        for person in people:
            for org in orgs:
                relationships.append({
                    'person': person.text,
                    'organization': org.text,
                    'sentence': sent.text.strip()
                })
    
    return relationships

def extract_location_info(text):
    """Extract location-based information"""
    doc = nlp(text)
    
    location_info = []
    sentences = list(doc.sents)
    
    for sent in sentences:
        locations = [ent for ent in sent.ents if ent.label_ in ["GPE", "LOC"]]
        if locations:
            location_info.append({
                'locations': [loc.text for loc in locations],
                'sentence': sent.text.strip()
            })
    
    return location_info

# Test with sample text
news_text = """
Tim Cook, the CEO of Apple Inc., announced new products at the company's headquarters in Cupertino, California.
Microsoft's CEO Satya Nadella spoke at the conference in Seattle, Washington.
The event was held on March 15, 2024, and attracted thousands of attendees from around the world.
"""

print("Person-Organization Relationships:")
print("=" * 40)
relationships = extract_person_org_relationships(news_text)
for rel in relationships:
    print(f"{rel['person']} -> {rel['organization']}")
    print(f"  Context: {rel['sentence']}")
    print()

print("Location Information:")
print("=" * 25)
location_info = extract_location_info(news_text)
for info in location_info:
    print(f"Locations: {', '.join(info['locations'])}")
    print(f"Context: {info['sentence']}")
    print()


In [ ]:
def extract_temporal_info(text):
    """Extract temporal information from text"""
    doc = nlp(text)
    
    temporal_info = []
    for ent in doc.ents:
        if ent.label_ == "DATE":
            # Find the sentence containing this date
            for sent in doc.sents:
                if ent.start >= sent.start and ent.end <= sent.end:
                    temporal_info.append({
                        'date': ent.text,
                        'sentence': sent.text.strip()
                    })
                    break
    
    return temporal_info

def create_fact_extractor():
    """Create a comprehensive fact extractor"""
    def extract_facts(text):
        doc = nlp(text)
        
        facts = {
            'entities': extract_key_entities(text),
            'relationships': extract_person_org_relationships(text),
            'locations': extract_location_info(text),
            'temporal': extract_temporal_info(text),
            'noun_phrases': extract_noun_phrases(text)
        }
        
        return facts
    
    return extract_facts

# Test the comprehensive fact extractor
fact_extractor = create_fact_extractor()
facts = fact_extractor(news_text)

print("Comprehensive Fact Extraction:")
print("=" * 35)
print(f"Total entities found: {sum(len(v) for v in facts['entities'].values())}")
print(f"Relationships found: {len(facts['relationships'])}")
print(f"Location mentions: {len(facts['locations'])}")
print(f"Temporal references: {len(facts['temporal'])}")
print(f"Key noun phrases: {len(facts['noun_phrases'])}")


## Project 3: Text Preprocessing Pipeline

### Overview
Build a comprehensive text preprocessing pipeline that:
- Cleans and normalizes text
- Removes noise and irrelevant content
- Extracts meaningful features
- Prepares text for further analysis


In [ ]:
# Text Preprocessing Pipeline
class TextPreprocessor:
    def __init__(self, nlp_model):
        self.nlp = nlp_model
    
    def clean_text(self, text):
        """Basic text cleaning"""
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove special characters but keep basic punctuation
        text = re.sub(r'[^\w\s.,!?;:]', '', text)
        return text.strip()
    
    def extract_meaningful_tokens(self, text):
        """Extract meaningful tokens (non-stop words, non-punctuation)"""
        doc = self.nlp(text)
        meaningful_tokens = []
        
        for token in doc:
            if (not token.is_stop and 
                not token.is_punct and 
                not token.is_space and
                len(token.text) > 1):
                meaningful_tokens.append({
                    'text': token.text,
                    'lemma': token.lemma_,
                    'pos': token.pos_,
                    'is_alpha': token.is_alpha
                })
        
        return meaningful_tokens
    
    def extract_features(self, text):
        """Extract various text features"""
        doc = self.nlp(text)
        
        features = {
            'word_count': len([token for token in doc if not token.is_space]),
            'sentence_count': len(list(doc.sents)),
            'avg_word_length': sum(len(token.text) for token in doc if not token.is_space) / max(1, len([token for token in doc if not token.is_space])),
            'pos_distribution': Counter([token.pos_ for token in doc if not token.is_space]),
            'entity_count': len(doc.ents),
            'entity_types': Counter([ent.label_ for ent in doc.ents])
        }
        
        return features
    
    def preprocess(self, text):
        """Complete preprocessing pipeline"""
        # Clean the text
        cleaned_text = self.clean_text(text)
        
        # Extract meaningful tokens
        tokens = self.extract_meaningful_tokens(cleaned_text)
        
        # Extract features
        features = self.extract_features(cleaned_text)
        
        return {
            'cleaned_text': cleaned_text,
            'tokens': tokens,
            'features': features
        }

# Test the preprocessing pipeline
preprocessor = TextPreprocessor(nlp)

sample_dirty_text = """
    This is a SAMPLE text with lots of    extra spaces and special characters!!! 
    It contains entities like Apple Inc. and dates like March 15, 2024.
    The text has some noise and irrelevant content that needs cleaning.
"""

result = preprocessor.preprocess(sample_dirty_text)

print("Text Preprocessing Results:")
print("=" * 30)
print(f"Cleaned text: {result['cleaned_text']}")
print(f"\nWord count: {result['features']['word_count']}")
print(f"Sentence count: {result['features']['sentence_count']}")
print(f"Average word length: {result['features']['avg_word_length']:.2f}")
print(f"Entity count: {result['features']['entity_count']}")
print(f"Entity types: {dict(result['features']['entity_types'])}")


In [ ]:
# Advanced preprocessing with batch processing
def batch_preprocess(texts, preprocessor):
    """Process multiple texts efficiently"""
    results = []
    
    # Process texts in batch for efficiency
    docs = list(preprocessor.nlp.pipe(texts))
    
    for i, doc in enumerate(docs):
        result = preprocessor.preprocess(texts[i])
        results.append(result)
    
    return results

# Test batch processing
sample_texts = [
    "Apple Inc. is a technology company founded in 1976.",
    "Microsoft Corporation develops software and cloud services.",
    "Google LLC provides internet-related services and products."
]

batch_results = batch_preprocess(sample_texts, preprocessor)

print("Batch Processing Results:")
print("=" * 30)
for i, result in enumerate(batch_results, 1):
    print(f"Text {i}:")
    print(f"  Entities: {result['features']['entity_count']}")
    print(f"  Words: {result['features']['word_count']}")
    print(f"  Entity types: {dict(result['features']['entity_types'])}")
    print()


## Summary and Key Takeaways

### What We Built

1. **Text Summarization Helper**
   - Entity extraction and categorization
   - Noun phrase identification
   - Sentence scoring based on entity density
   - Key information prioritization

2. **Information Extraction System**
   - Person-organization relationship mapping
   - Location-based information extraction
   - Temporal information identification
   - Comprehensive fact extraction

3. **Text Preprocessing Pipeline**
   - Text cleaning and normalization
   - Meaningful token extraction
   - Feature extraction and analysis
   - Batch processing capabilities

### Key Skills Demonstrated

- **Entity Recognition**: Extracting and categorizing named entities
- **Relationship Mapping**: Finding connections between entities
- **Text Analysis**: Understanding text structure and content
- **Feature Engineering**: Creating meaningful text features
- **Pipeline Design**: Building reusable processing workflows

### Real-World Applications

- **Content Analysis**: Understanding and categorizing text content
- **Information Retrieval**: Extracting structured data from unstructured text
- **Text Mining**: Discovering patterns and insights in text data
- **Document Processing**: Automating text analysis workflows
- **Search and Recommendation**: Improving search with entity recognition

### Best Practices Learned

1. **Modular Design**: Break complex tasks into smaller, reusable functions
2. **Error Handling**: Consider edge cases and data quality issues
3. **Performance**: Use batch processing for efficiency
4. **Documentation**: Document functions and their purposes
5. **Testing**: Validate results with sample data

### Next Steps

- **Scale Up**: Apply these techniques to larger datasets
- **Integration**: Combine with machine learning models
- **Customization**: Adapt for specific domains or languages
- **Deployment**: Create production-ready applications
- **Advanced Features**: Add sentiment analysis, topic modeling, etc.

### Common Challenges

1. **Data Quality**: Handle noisy or incomplete text
2. **Performance**: Optimize for large-scale processing
3. **Accuracy**: Validate entity recognition results
4. **Domain Specificity**: Adapt to specialized text types
5. **Language Support**: Extend to multiple languages
